> **Historical artifacts:** Saved tables and figures predate wta-lcs-v2. Old sequence F1 is unordered set overlap; CV/F1/continuity need regeneration with the corrected protocol. See docs/EVALUATION_CORRECTIONS.md.

# PathSnap: Distributed Map-Matching & Route Reconstruction
**Course:** UE24CS352A: Machine Learning — PES University (RR Campus)  
**Team: SI No 107**
- **PES1UG24CS395** — Saatvik Gupta
- **PES1UG24CS396** — Sagar A Walikar
- **PES1UG24CS406** — Samarth Kotian
- **PES1UG24CS407** — Samarth M H

---
### Workflow Overview
1. **Exploratory Data Analysis & Local Tangent-Plane Coordinate Projection**
2. **Multi-Point `cKDTree` Spatial Candidate Generation ($K=8$)**
3. **25-Feature Spatial, Kinematic, Heading & Topological Engineering**
4. **Leakage-Free 5-Fold `GroupKFold` Benchmarking Across 7 Models (6 Algorithm Families)**
5. **Bayesian Hyperparameter Optimisation (`Optuna TPESampler`)**
6. **Road-Graph HMM & Log-Domain Viterbi Route Reconstruction**
7. **Interactive Folium Map Visualisation & Explainability (SHAP)**

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add project root to sys.path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import RANDOM_SEED, set_global_seed, EDA_OUT_DIR, RESULTS_OUT_DIR, FIGURES_OUT_DIR, MAPS_OUT_DIR
from src.data_loader import load_partition
from src.candidates import generate_candidates_for_trip
from src.features import FEATURE_NAMES, extract_features_for_trip
from src.visualization import generate_folium_map_for_trip
from predict import load_inference_artifacts, predict_single_trip

set_global_seed(RANDOM_SEED)
print(f'Project Root: {PROJECT_ROOT} | Seed: {RANDOM_SEED}')

## 1. Dataset Partition Summary & Exploratory Data Analysis
We inspect the 80 training trips (`train`) and 10 development-stage student-testing trips (`student_test`).

In [ ]:
part_df = pd.read_csv(os.path.join(EDA_OUT_DIR, 'partition_summary.csv'))
display(part_df)
display(Image(filename=os.path.join(EDA_OUT_DIR, 'eda_dataset_overview.png'), width=850))
display(Image(filename=os.path.join(EDA_OUT_DIR, 'eda_spatial_and_network.png'), width=850))

## 2. Multi-Family Model Comparison (7 Models Across 6 Algorithm Families)
All models are cross-validated using 5-fold `GroupKFold` grouped strictly by `trip_id` on the 80 training trips and evaluated on the 10 student-testing trips.

In [ ]:
models_df = pd.read_csv(os.path.join(RESULTS_OUT_DIR, 'model_comparison_results.csv'))
cols_show = [
    'model_name', 'algorithm_family', 'cv_macro_f1_mean', 'cv_macro_f1_std',
    'student_test_top1_accuracy', 'route_continuity', 'student_test_macro_f1',
    'student_test_log_loss', 'inference_ms_per_trip'
]
display(models_df[cols_show])
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'model_comparison_f1_top1.png'), width=850))
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'model_logloss_and_latency.png'), width=850))
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'confusion_matrices_grid.png'), width=850))
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'epoch_training_curves.png'), width=850))

## 3. Bayesian Hyperparameter Optimisation (Optuna TPE)
We tune LightGBM using Optuna's Tree-structured Parzen Estimator (`TPESampler`) over 9 hyperparameters.

In [ ]:
tuned_df = pd.read_csv(os.path.join(RESULTS_OUT_DIR, 'bayesian_untuned_vs_tuned.csv'))
display(tuned_df)
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'bayesian_optimisation_history.png'), width=850))
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'feature_importance_shap.png'), width=850))

## 4. HMM & Log-Domain Viterbi Route Reconstruction
Comparing pointwise Classifier-Only predictions against sequential Classifier + HMM/Viterbi decoding across all 10 student-testing trips.

In [ ]:
hmm_df = pd.read_csv(os.path.join(RESULTS_OUT_DIR, 'hmm_viterbi_ablation.csv'))
display(hmm_df)
display(Image(filename=os.path.join(FIGURES_OUT_DIR, 'hmm_viterbi_ablation.png'), width=850))

## 5. Geographical Route Reconstruction Maps (Folium & High-Res Views)

In [ ]:
display(Image(filename=os.path.join(MAPS_OUT_DIR, 'map_comparison_00000000.png'), width=880))
display(Image(filename=os.path.join(MAPS_OUT_DIR, 'map_comparison_00000052.png'), width=880))